In [1]:
import pandas as pd
df = pd.read_csv('data/syntheticallyGeneratedData/active_directory_tickets_258.csv')
print(df.columns.tolist())
print(df.head(3))

['text', 'label']
                                                text             label
0  Need help with account, it is locked out again...  Active Directory
1   account password expired since morning thank you  Active Directory
2  Please disable account for employee who resign...  Active Directory


In [2]:
import pandas as pd

files = {
    'Active Directory': 'data/syntheticallyGeneratedData/active_directory_tickets_258.csv',
    'Computer-Services': 'data/syntheticallyGeneratedData/computer_services_tickets.csv',
    'O365': 'data/syntheticallyGeneratedData/o365_tickets.csv',
    'Software': 'data/syntheticallyGeneratedData/software_tickets_211.csv',
    'Fileservice': 'data/syntheticallyGeneratedData/fileservice_tickets_133.csv',
}

for label, path in files.items():
    df = pd.read_csv(path)
    print(f"\n===== {label} =====")
    print(f"Shape: {df.shape}")
    print(f"Columns: {df.columns.tolist()}")
    print(df.head(3))
    print()


===== Active Directory =====
Shape: (258, 2)
Columns: ['text', 'label']
                                                text             label
0  Need help with account, it is locked out again...  Active Directory
1   account password expired since morning thank you  Active Directory
2  Please disable account for employee who resign...  Active Directory


===== Computer-Services =====
Shape: (274, 3)
Columns: ['id', 'category', 'ticket']
   id           category                                             ticket
0   1  Computer-Services  my laptop wont turn on at all just a black scr...
1   2  Computer-Services  printer in room 204 is jammed again third time...
2   3  Computer-Services  need a monitor ordered for the new guy startin...


===== O365 =====
Shape: (212, 2)
Columns: ['text', 'label']
                                                text label
0  Outlook not opening just shows a loading scree...  O365
1   Teams keeps crashing mid call really frustrating  O365
2  my onedrive

In [3]:
import pandas as pd
import os

# load original training data
train = pd.read_csv('data/train_final.csv')
print("Original train shape:", train.shape)
print(train['label'].value_counts())

# load and normalize each synthetic file
dfs = [train[['text_clean', 'label']].rename(columns={'text_clean': 'text'})]

# active directory
ad = pd.read_csv('data/syntheticallyGeneratedData/active_directory_tickets_258.csv')
dfs.append(ad[['text', 'label']])

# computer services — different column names
cs = pd.read_csv('data/syntheticallyGeneratedData/computer_services_tickets.csv')
cs = cs.rename(columns={'ticket': 'text', 'category': 'label'})
dfs.append(cs[['text', 'label']])

# o365
o365 = pd.read_csv('data/syntheticallyGeneratedData/o365_tickets.csv')
dfs.append(o365[['text', 'label']])

# software
sw = pd.read_csv('data/syntheticallyGeneratedData/software_tickets_211.csv')
dfs.append(sw[['text', 'label']])

# fileservice
fs = pd.read_csv('data/syntheticallyGeneratedData/fileservice_tickets_133.csv')
dfs.append(fs[['text', 'label']])

# combine
train_augmented = pd.concat(dfs, ignore_index=True)
train_augmented = train_augmented.rename(columns={'text': 'text_clean'})

print("\n===== AUGMENTED DATASET =====")
print("Total shape:", train_augmented.shape)
print(train_augmented['label'].value_counts())

# save
train_augmented.to_csv('data/train_augmented.csv', index=False)
print("\nSaved to data/train_augmented.csv")

Original train shape: (928, 7)
label
Support general      336
Fileservice          203
Software             125
O365                 124
Active Directory      78
Computer-Services     62
Name: count, dtype: int64

===== AUGMENTED DATASET =====
Total shape: (2016, 2)
label
Fileservice          336
Support general      336
Software             336
O365                 336
Active Directory     336
Computer-Services    336
Name: count, dtype: int64

Saved to data/train_augmented.csv


In [5]:
import pandas as pd
import pickle
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# load data
train_aug = pd.read_csv('data/train_augmented.csv')
test = pd.read_csv('data/test_final.csv')

print("Train shape:", train_aug.shape)
print("Test shape:", test.shape)

# load sentence transformer
st_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

print("\nEncoding augmented train...")
X_tr = st_model.encode(train_aug['text_clean'].tolist(), show_progress_bar=True)
print("Encoding test...")
X_te = st_model.encode(test['text_clean'].tolist(), show_progress_bar=True)

y_tr = train_aug['label'].tolist()
y_te = test['label'].tolist()

# train
clf_aug = LogisticRegression(max_iter=1000, class_weight='balanced')
clf_aug.fit(X_tr, y_tr)

# evaluate
y_pred = clf_aug.predict(X_te)

print("\n===== AUGMENTED MODEL RESULTS =====")
print(classification_report(y_te, y_pred))

# save model
with open('models/emb_augmented_classifier.pkl', 'wb') as f:
    pickle.dump(clf_aug, f)
print("Model saved.")

Train shape: (2016, 2)
Test shape: (599, 4)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3803.31it/s]



Encoding augmented train...


Batches: 100%|██████████| 63/63 [00:07<00:00,  8.47it/s]


Encoding test...


Batches: 100%|██████████| 19/19 [00:03<00:00,  5.36it/s]



===== AUGMENTED MODEL RESULTS =====
                   precision    recall  f1-score   support

 Active Directory       0.65      0.41      0.50        49
Computer-Services       0.89      0.76      0.82        41
      Fileservice       0.95      0.96      0.96       138
             O365       0.72      0.63      0.67        92
         Software       0.66      0.53      0.59        58
  Support general       0.74      0.88      0.80       221

         accuracy                           0.78       599
        macro avg       0.77      0.70      0.72       599
     weighted avg       0.78      0.78      0.77       599

Model saved.
